# Lab 1 · Regression: asphaltene precipitation
**Slides:** Module 06 (algorithms) and 07 (evaluation)  ·  **Time:** 75 min

Adding an n-alkane (n-C5 … n-C12) to crude oil precipitates asphaltene. We predict the **weight percent precipitated**
from temperature, the alkane carbon number and the dilution ratio (mL alkane per g oil), and compare ten algorithms,
the same ones as the course notebooks (`Linear_Regression`, `KNN`, `SVR`, `DT`, `RF`, `AdaBoost`, `Gradient_Boosting`, `NN`, `Voting`).

**You will:** compare algorithms · tune with cross-validation instead of the test set · test on a whole unseen series · explain the model.

In [ ]:
# --- setup: works locally and in Google Colab ---
import os, sys
if "google.colab" in sys.modules and not os.path.exists("aiml"):
    print("Upload the course kit zip to Colab, then unzip it:  !unzip -q aiml_course_kit.zip && %cd aiml_course_kit")
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("."))
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3, "axes.spines.top": False, "axes.spines.right": False})
DATA = "../data" if os.path.exists("../data") else "data"

from aiml.data import load_asphaltene, ASPH_FEATURES, ASPH_TARGET
from aiml.models import regressors
from aiml.evaluate import compare, crossplot, hyper_curve, report, importance
from sklearn.model_selection import train_test_split, cross_val_score, KFold, GroupKFold, cross_val_predict
from sklearn.metrics import r2_score
df = load_asphaltene()
X, y = df[ASPH_FEATURES], df[ASPH_TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 1 · Ten algorithms on all 167 tests

In [ ]:
res_all = compare(regressors(), X_train, y_train, X_test, y_test); res_all[["model", "R2_train", "R2_test", "MAE_test", "fit_s"]]

## 2 · The same comparison without the 17 suspect tests (Lab 0)

In [ ]:
clean = load_asphaltene(drop_suspect=True)
Xc, yc = clean[ASPH_FEATURES], clean[ASPH_TARGET]
Xc_train, Xc_test, yc_train, yc_test = train_test_split(Xc, yc, test_size=0.2, random_state=42)
res_clean = compare(regressors(), Xc_train, yc_train, Xc_test, yc_test)
res_all[["model", "R2_test"]].merge(res_clean[["model", "R2_test"]], on="model", suffixes=("_all", "_clean"))

Seventeen rows decide the ranking. With them, only flexible models (trees, neural nets) can chase the three extreme
series and linear regression fails; without them, even linear regression reaches R² ≈ 0.96. **Clean data first,
then choose the algorithm.** We continue with the clean data.

## 3 · Tuning: the course loops, done safely
The course notebooks loop over `max_depth` or `n_neighbors` and pick the value with the best **test** score. That uses
the test set to make a decision, so the test score is no longer an honest estimate. Use cross-validation on the
training data to tune, and touch the test set once at the end.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
hyper_curve(RandomForestRegressor(n_estimators=100, random_state=0), "max_depth", list(range(1, 15)), Xc_train, yc_train, Xc_test, yc_test, ax=ax[0])
cv = KFold(5, shuffle=True, random_state=0)
ks = list(range(1, 16))
cv_scores = [cross_val_score(make_pipeline(StandardScaler(), KNeighborsRegressor(k)), Xc_train, yc_train, cv=cv).mean() for k in ks]
ax[1].plot(ks, cv_scores, "o-"); ax[1].set_xlabel("n_neighbors"); ax[1].set_ylabel("5-fold CV R² (training data only)")
ax[1].set_title(f"KNN: best k by cross-validation = {ks[int(np.argmax(cv_scores))]}"); plt.tight_layout(); plt.show()

### ✏️ Your turn 1
Tune `C` of the SVR (`regressors()["SVR"]`, parameter name `svr__C`) over `[0.3, 1, 3, 10, 30, 100, 300]` with 5-fold
cross-validation on the training data. Then fit the best one and report the test score once.

In [ ]:
svr = regressors()["SVR"]
Cs = [0.3, 1, 3, 10, 30, 100, 300]
# cross_val_score(svr.set_params(svr__C=c), Xc_train, yc_train, cv=cv).mean()

## 4 · A harder, more honest test: unseen series
A random split puts points of the same curve in train and test, so the model only interpolates along a curve it
already saw. In practice you want to predict a **new temperature or a new precipitant**. `GroupKFold` holds out whole
series (one temperature + n-alkane curve at a time).

In [ ]:
groups = clean["series"]
rows = []
for name, m in regressors().items():
    p_rand = cross_val_predict(m, Xc, yc, cv=KFold(5, shuffle=True, random_state=0))
    p_grp = cross_val_predict(m, Xc, yc, cv=GroupKFold(5), groups=groups)
    rows.append({"model": name, "random 5-fold R²": r2_score(yc, p_rand), "unseen-series R²": r2_score(yc, p_grp)})
pd.DataFrame(rows).sort_values("unseen-series R²", ascending=False).round(3)

### ✏️ Your turn 2 · physics features
Precipitation grows roughly with the **logarithm** of dilution. Add `log_dilution` (`load_asphaltene(drop_suspect=True, add_features=True)`)
and repeat the unseen-series test for *Linear regression*, *KNN* and *Neural network (MLP)*. Which model gains most, and why?

In [ ]:
cf = load_asphaltene(drop_suspect=True, add_features=True)
F2 = ASPH_FEATURES + ["log_dilution"]

## 5 · Final model, cross-plot and explanation

In [ ]:
best = regressors()["Gradient boosting"].fit(Xc_train, yc_train)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
crossplot(yc_train, best.predict(Xc_train), yc_test, best.predict(Xc_test), "Gradient boosting · asphaltene", "wt%", ax=ax[0])
imp = importance(best, Xc_test, yc_test, ax=ax[1]); ax[1].set_title("permutation importance (test data)"); plt.tight_layout(); plt.show()

In [ ]:
# what-if: precipitation curves the model predicts for n-heptane at four temperatures
dr = np.linspace(3, 25, 60)
for T in [293, 308, 323, 332]:
    plt.plot(dr, best.predict(pd.DataFrame({"temperature_K": T, "alkane_carbon_number": 7, "dilution_ratio": dr})), label=f"{T} K")
plt.xlabel("dilution ratio (mL/g)"); plt.ylabel("predicted wt%"); plt.title("model response · n-C7"); plt.legend(); plt.show()

### ✏️ Your turn 3
Look at the 332 K curve. Is it smooth and physical between the points the model saw? Tree ensembles predict in **steps**;
try `regressors()["Neural network (MLP)"]` or `"SVR"` and compare the shape. Which would you trust for a new dilution ratio?

## Key ideas
* Data quality (17 rows) changed the winner more than any algorithm choice.
* Tune with cross-validation on training data; use the test set once.
* Split the way the model will be used: unseen series is harder and more honest than a random split.
* A simple physics feature (log dilution) can make a simple model as good as a complex one.

In [ ]:
# compare the shape of the response curves for another model